<a href="https://colab.research.google.com/github/ElofssonLab/kb8029-book/blob/main/notebooks/session05-discussion-2.ipynb" style="display:inline-block;padding:10px 18px;background-color:#F9AB00;color:#000000;font-weight:bold;text-decoration:none;border-radius:6px;font-family:sans-serif;font-size:14px;">&#9654;&nbsp; Open in Google Colab</a>

# Session 5: in-class discussion problem (2 of 3)

This is the block 2 question of the lecture quiz. First guess the answer in the quiz, on your own.
Then fix and run the code below, discuss the outcome with your neighbour, and answer the quiz again.

## What does a residue that was never seen do to the score?

A PSSM is built from five aligned proteins:

```
Position:   1 2 3 4 5
Protein 1:  A G L S P
Protein 2:  A G L T P
Protein 3:  R G I S P
Protein 4:  A A L S Q
Protein 5:  A A L S P
```

with a pseudocount of +1 added to every residue at every position. A new sequence, `GGLSP`, has a
G at position 1, where none of the five proteins has one. What happens to its score, compared
with the consensus `AGLSP`?

**Fix and run.** The cell below builds the PSSM and scores three sequences, but the line marked
`FIX` leaves the pseudocount out. Run it once as it is: what score does `GGLSP` get? Then put the
pseudocount back and run it again.

In [1]:
import numpy as np

msa = ["AGLSP", "AGLTP", "RGISP", "AALSQ", "AALSP"]
background = {"A": 0.074, "G": 0.074, "I": 0.038, "L": 0.076, "P": 0.050,
              "Q": 0.037, "R": 0.042, "S": 0.081, "T": 0.062}   # a toy alphabet of 9 residues
residues = sorted(background)

pseudocount = 0                  # FIX: the PSSM should add +1 to every count

pssm = []
for pos in range(len(msa[0])):
    column = [seq[pos] for seq in msa]
    total = len(column) + pseudocount * len(residues)
    with np.errstate(divide="ignore"):   # log2(0) gives -inf, without a warning
        pssm.append({r: np.log2((column.count(r) + pseudocount) / total / background[r])
                     for r in residues})

def score(seq):
    return sum(pssm[i][r] for i, r in enumerate(seq))

for seq in ["AGLSP", "GGLSP", "RGISP"]:
    print(f"{seq}: {score(seq):6.2f}")
print(f"\nG at position 1: {pssm[0]['G']:.2f}    A at position 1: {pssm[0]['A']:.2f}")

AGLSP:  17.15
GGLSP:   -inf
RGISP:  14.97

G at position 1: -inf    A at position 1: 3.43


**If you finish early.**

1. Try a pseudocount of 0.1 instead of 1. What happens to G's score at position 1?
2. With +1 for each of 9 residues, how many pseudocounts are added to each column, and how many
   real counts are there? Which of the two dominates the profile?

**Discussion point.** Without pseudocounts, G at position 1 has frequency 0, and
$\log_2 0 = -\infty$: one unseen residue rules out the whole sequence, however well the rest of it
fits. Five sequences are far too few to conclude that G can never occur there.

With +1 pseudocounts, `GGLSP` scores 9.11 against 11.43 for the consensus `AGLSP`: lower, but
finite and still clearly positive. G's score at position 1 is only −0.05, close to neutral, because
the 9 pseudocounts outweigh the 5 real sequences in each column; A's score there drops from 3.43 to
2.27 for the same reason. A pseudocount is a prior: it says what we believe before seeing the data,
and with few sequences the prior speaks loudly. Real tools add pseudocounts in proportion to the
background frequencies or to substitution scores, and give them less weight as the alignment grows.